# 01. BaseModel Essentials & Type Coercion (Lax vs. Strict)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/en/01_basemodel_quickstart.ipynb)

Pydantic v2 turns standard Python type annotations (`class User(BaseModel):`) into a blazing-fast, Rust-powered validation and serialization pipeline backed by `pydantic-core`. Because `pydantic` v2 comes pre-installed in Google Colab and requires zero API keys or external services, you can run every cell in this cookbook immediately out of the box.

![Figure 1-1: Pydantic v2 End-to-End Validation & Serialization Pipeline](https://examples.drawlib.com/pydantic/images/01_basemodel_quickstart/ch01_pydantic_pipeline.png)

*Figure 1-1: Pydantic v2 End-to-End Validation & Serialization Pipeline*

## 1. Your First `BaseModel` & Automatic Lax Type Coercion

To define a schema in Pydantic v2, subclass `BaseModel` and declare fields using standard Python type hints (`int`, `str`, `bool`, `list[str]`). By default, Pydantic runs in **Lax Mode**: when URL query parameters, form inputs, or loosely typed payloads pass strings such as `"42"` or `"true"`, `pydantic-core` automatically coerces compatible values into native Python types while guaranteeing runtime type safety:

In [ ]:
!pip install -q pydantic

In [ ]:
from pydantic import BaseModel


class User(BaseModel):
    id: int
    name: str
    is_active: bool = True
    tags: list[str] = []


user = User(id="42", name="Alice", is_active="true", tags=["admin", "colab"])

print(user)
print("type(user.id):", type(user.id), "| is_active:", user.is_active)
print("model_dump():", user.model_dump())

Notice also how Pydantic v2 standardizes all validation and serialization methods under the `model_*` prefix—replacing deprecated v1 names (`parse_obj`, `parse_raw`, `dict`, and `json`):

- **`Model.model_validate(obj)`** *(v1: `parse_obj`)*: Validates a Python `dict` or object into a typed `BaseModel` instance.
- **`Model.model_validate_json(data)`** *(v1: `parse_raw`)*: Parses and validates raw UTF-8 JSON strings or `bytes` directly in Rust (`jiter`) in a single pass—skipping Python's `json.loads()` overhead.
- **`instance.model_dump()` / `instance.model_dump_json()`** *(v1: `dict()` / `json()`)*: Serializes the validated model back into a Python `dict` or formatted JSON string.

In [ ]:
raw_json = '{"id": "108", "name": "Bob", "tags": ["data", "rust"]}'

bob = User.model_validate_json(raw_json)
print("Validated from JSON:", bob)
print(bob.model_dump_json(indent=2))

## 2. Lax Mode vs. Strict Mode (`ConfigDict(strict=True)`)

While automatic coercion in **Lax Mode** is ideal at application boundaries (HTTP query strings, environment variables, CSV imports), internal service-to-service APIs and financial pipelines often want to reject mismatched wire types instead of silently converting `"42"` to `42`.

Pydantic v2 lets you switch to **Strict Mode** either **per validation call** (`strict=True`) or **across an entire model** (`model_config = ConfigDict(strict=True)`):

- **Lax Mode (Default)**: Converts lossless/unambiguous inputs (`"42"` → `42`, `"true"` → `True`, `[1, 2]` → `(1, 2)`), while still rejecting unparseable inputs like `"forty-two"`.
- **Strict Mode (`strict=True`)**: Requires exact type identity in Python (`"42"` fails `int` with `int_type`, `"true"` fails `bool` with `bool_type`, and `[1, 2]` fails `tuple[int, int]` with `tuple_type` when validating Python dicts).

![Figure 1-2: Lax Mode (Default Coercion) vs. Strict Mode Exact Type Matching](https://examples.drawlib.com/pydantic/images/01_basemodel_quickstart/ch01_lax_vs_strict.png)

*Figure 1-2: Lax Mode (Default Coercion) vs. Strict Mode Exact Type Matching*

Let's compare Lax vs. Strict validation side-by-side on our `User` model—first passing `strict=True` to `User.model_validate(...)`, and then locking strictness onto a dedicated model class via `ConfigDict(strict=True)`:

In [ ]:
from pydantic import ConfigDict, ValidationError

payload = {"id": "42", "name": "Alice"}

# 1. Default Lax Mode: coerces "42" -> 42
lax_user = User.model_validate(payload)
print("Lax mode succeeded:", repr(lax_user))

# 2. Per-call Strict Mode: rejects string "42" for int field
try:
    User.model_validate(payload, strict=True)
except ValidationError as e:
    print("\nStrict mode raised ValidationError:")
    print(e)


# 3. Model-wide Strict Mode via ConfigDict(strict=True)
class StrictPoint(BaseModel):
    model_config = ConfigDict(strict=True)

    label: str
    coords: tuple[int, int]


point = StrictPoint(label="origin", coords=(0, 0))
print("\nStrictPoint exact match:", repr(point))

Notice how `pydantic-core` pinpoints the exact offending field (`id`), human-readable message (`Input should be a valid integer`), error code (`type=int_type`), and raw `input_value='42'`—and when you parse JSON via `StrictPoint.model_validate_json('{"label": "origin", "coords": [0, 0]}')`, Strict Mode still accepts JSON arrays for `tuple` fields because JSON has no native tuple literal.

### Key Takeaways & Next Step

- **`BaseModel` + v2 `model_*` API**: Declare fields with standard Python type hints, validate dicts or raw JSON strings at Rust speed via `model_validate()` / `model_validate_json()`, and serialize cleanly with `model_dump()` / `model_dump_json()`.
- **Up Next in Chapter 02**: Learn how to attach declarative numeric, string, and safe factory defaults with `Field(...)` and programmatically inspect structured multi-field `ValidationError` payloads (`e.errors()`).